# 02 · Evaluate & submit / 평가와 제출

Copy의 공개 Eval Micro CER **11.766335975170426%**를 재현하고, 01에서 만든 학습 모델 또는 자신의 CSV를 비교합니다.
채점만 실행할 때는 음성 ZIP이나 GPU가 필요하지 않습니다. 웹과 동일한 Python 평가 규격을 사용합니다.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/train_starter.py").is_file()),
            Path.cwd() / "TURST-YOUR-EARS")
if not (repo / "scripts/train_starter.py").is_file():
    subprocess.run(["git", "clone", "https://github.com/jhparktime/TURST-YOUR-EARS.git", str(repo)], check=True)
os.chdir(repo)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Python:", sys.version.split()[0])
def run(*args):
    subprocess.run([sys.executable, "-m", *map(str, args)], check=True)
data = repo / "datasets/tye-ko-asr-v2"
work = repo / "starter-output"
work.mkdir(exist_ok=True)


## 1. Copy 기준점 재현

In [ ]:
run("scripts.baseline", "--mode", "copy", "--inputs", data / "eval/inputs.json",
    "--output", work / "eval-copy.csv")
run("scripts.evaluate", "--references", data / "eval/references.json",
    "--predictions", work / "eval-copy.csv", "--output", work / "copy.scores.json")
copy = json.loads((work / "copy.scores.json").read_text())
assert abs(copy["micro_cer"] - 11.766335975170426) < 1e-10
print("Copy Micro CER:", copy["micro_cer"])


## 2. 내 CSV 평가
`prediction_path`를 자신의 UTF-8 CSV 경로로 바꾸세요. 헤더는 `id,prediction`, 공개 Eval 409개 ID가 각각 한 번씩 있어야 합니다.
01을 아직 실행하지 않았다면 위에서 생성한 Copy CSV로 동작을 확인합니다.


In [ ]:
prediction_path = work / "eval-token.csv"
if not prediction_path.is_file():
    prediction_path = work / "eval-copy.csv"
run("scripts.evaluate", "--references", data / "eval/references.json",
    "--predictions", prediction_path, "--output", work / "my.scores.json")
mine = json.loads((work / "my.scores.json").read_text())
for key in ["micro_cer", "macro_cer", "micro_wer", "cer_improvement_pp", "slot_restoration_rate", "harmful_edit_rate"]:
    print(f"{key}: {mine[key]:.6f}")
print("Copy 대비 CER 개선(%p):", copy["micro_cer"] - mine["micro_cer"])


## 3. 웹에서 같은 CSV 채점
[Trust Your Ears 웹 채점](https://jhparktime.github.io/TURST-YOUR-EARS/#submit)에 CSV를 선택하세요.
파일은 브라우저에서 채점되며 공개 순위에 자동 등록되지 않습니다. JSON을 저장해 실험 기록에 남기세요.
Eval 정답은 개발용이며 추론 입력으로 넣지 않습니다. [참가 규칙](https://github.com/jhparktime/TURST-YOUR-EARS/blob/main/docs/competition-rules.md)을 확인하세요.
